In [1]:
# ─────────────────────────────────────────────────────────────────────
# NOTEBOOK 02 · CELL 1 · SETUP AND LOAD THE CLEAN TABLES FROM NOTEBOOK 01
# Purpose: start from the cleaned data (no cleaning is repeated) and set the rules for this notebook.
# ─────────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

OUT_DIR = Path("interim")      # same folder notebook 01 saved into
SEED = 42
np.random.seed(SEED)

# TECHNICAL: notebook 01 saved parquet if `pyarrow` exists, otherwise pickle. This helper reads whichever is there.
def load_table(name):
    if (OUT_DIR / f"{name}.parquet").exists():
        return pd.read_parquet(OUT_DIR / f"{name}.parquet")
    return pd.read_pickle(OUT_DIR / f"{name}.pkl")

ball    = load_table("ball_clean")
matches = load_table("matches_clean")
players = load_table("players")
teams   = load_table("teams")
squads  = load_table("squads")
xi      = load_table("playing_eleven")

# FUNCTIONAL: the single most important rule of this notebook.
# Every feature we build describes the game AS IT STANDS BEFORE THE BALL IS BOWLED. A coach has to decide with
# only that information; a model that "knows" the outcome of the ball (or later balls) would look brilliant
# and be useless. This is called LEAKAGE, and Cell 8 tests that we avoided it.
# TECHNICAL: a "before this ball" value is built as  cumulative sum INCLUDING this ball  minus  this ball's own value.

# Deliveries must be in playing order for every running count below.
ball = ball.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
print(f"ball: {ball.shape} | matches: {matches.shape} | players: {players.shape}")
display(pd.DataFrame({"rows": [len(ball), len(matches), len(players), len(teams), len(squads), len(xi)]},
                      index=["ball", "matches", "players", "teams", "squads", "playing_eleven"]).T)
print("Innings in ball table:", sorted(ball.innings.unique()), "| tiers:", ball.tier.value_counts().to_dict())

ball: (200961, 40) | matches: (883, 36) | players: (959, 16)


,ball,matches,players,teams,squads,playing_eleven
rows,200961,883,959,353,6413,28381


Innings in ball table: [1, 2] | tiers: {'T20I': 106315, 'IPL': 87673, 'A-team / emerging': 6973}


### Cell 1: Setup and load
**Procedure:** Loaded the cleaned tables saved by notebook 01 (no cleaning is repeated), fixed the seed, and sorted deliveries into playing order. Set the one rule for this notebook: every feature describes the game **before** the ball is bowled, so nothing from the ball's outcome or later balls can leak in.

**Observations**
- Tables load correctly: 200,961 deliveries, 883 matches, 959 players, 353 teams, 6,413 squad rows and 28,381 playing-XI rows. These match the notebook 01 save exactly.
- Only innings 1 and 2 are present (super overs are gone), and the tier split is T20I 106,315, IPL 87,673 and A-team 6,973.

In [2]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 · MATCH STATE BEFORE EACH BALL: score, wickets, balls left, over, phase
# Purpose: the basic scoreboard a coach reads before every delivery.
# ─────────────────────────────────────────────────────────────────────
inn = ball.groupby(["match_id", "innings"])          # TECHNICAL: one group per innings; cumulative sums restart in each

# 2a) Running counts BEFORE the ball (cumulative sum minus this ball's own contribution).
ball["legal_before"] = inn.is_legal.cumsum() - ball.is_legal              # legal balls already bowled
ball["score_before"] = inn.total_runs.cumsum() - ball.total_runs          # team runs already scored
ball["wkts_before"]  = inn.is_wicket_clean.cumsum() - ball.is_wicket_clean  # wickets already lost

# 2b) How many legal balls does this innings have? FUNCTIONAL: normally 120, but rain can cut an innings
# (DLS). For those matches the data gives the revised number of balls, and we must use it, otherwise "balls left"
# would be too high and the chase would look easier than it was.
lim = matches.set_index("match_id")[["first_inn_revised_balls", "second_inn_revised_balls"]]
rev = np.where(ball.innings == 1, ball.match_id.map(lim.first_inn_revised_balls), ball.match_id.map(lim.second_inn_revised_balls))
ball["innings_balls"] = np.where(pd.isna(rev), 120, rev).astype(int)
ball["balls_left"]    = ball.innings_balls - ball.legal_before
print("Innings with a rain-shortened ball limit:", ball.loc[ball.innings_balls < 120, ["match_id", "innings"]].drop_duplicates().shape[0])

# 2c) Over number and phase. The data's `overid` is the over (1-20). We verify it against our own count first.
ours = ball.legal_before // 6 + 1
print(f"`overid` agrees with our legal-ball count on {(ours == ball.overid).mean():.2%} of deliveries "
      "(differences are no-ball/wide re-bowls at over boundaries), so we use `overid`.")
ball["over"] = ball.overid
# FUNCTIONAL: T20 has three natural phases: Powerplay (overs 1-6, only two fielders outside the circle),
# Middle (7-15, spinners and consolidation) and Death (16-20, all-out hitting). Pressure and tactics differ by phase.
ball["phase"] = pd.cut(ball.over, [0, 6, 15, 20], labels=["powerplay", "middle", "death"]).astype(str)

# 2d) Current run rate (CRR): runs per over so far. Undefined before the first legal ball, so we leave it missing
# (TECHNICAL: gradient-boosted models handle missing values natively; inventing a number would mislead them).
ball["crr"] = np.where(ball.legal_before > 0, ball.score_before * 6 / ball.legal_before.replace(0, np.nan), np.nan)

# 2e) Sanity: first ball of each innings must start at 0 runs, 0 wickets, full balls left.
first = ball[ball.innings_ball_num == 1]
assert (first.score_before == 0).all() and (first.wkts_before == 0).all() and (first.legal_before == 0).all()
print("First-ball state is 0/0/0 in every innings: OK")
display(ball.groupby("phase").agg(deliveries=("match_id", "size"), runs_per_ball=("total_runs", "mean"),
                                  wicket_rate=("is_wicket_clean", "mean")).loc[["powerplay", "middle", "death"]])
display(ball[["match_id", "innings", "over", "legal_before", "score_before", "wkts_before", "balls_left", "crr", "phase"]].iloc[[0, 1, 2, 60, 119]])

Innings with a rain-shortened ball limit: 48
`overid` agrees with our legal-ball count on 99.99% of deliveries (differences are no-ball/wide re-bowls at over boundaries), so we use `overid`.
First-ball state is 0/0/0 in every innings: OK


,deliveries,runs_per_ball,wicket_rate
phase,,,
powerplay,64008,1.388,0.045
middle,92499,1.342,0.048
death,44454,1.601,0.081


,match_id,innings,over,legal_before,score_before,wkts_before,balls_left,crr,phase
0,13511,1,1,0,0,0,120,NaN,powerplay
1,13511,1,1,1,0,0,119,0.000,powerplay
2,13511,1,1,2,0,0,118,0.000,powerplay
60,13511,1,10,57,37,5,63,3.895,middle
119,13511,1,20,116,103,8,4,5.328,death


### Cell 2: Match state before each ball
**Procedure:** For every delivery, computed legal balls bowled, team score and wickets lost *before* the ball (cumulative sum minus the ball's own value), the innings ball limit (120, or the revised figure in rain-affected games), balls left, over, phase (powerplay 1–6, middle 7–15, death 16–20) and current run rate.

**Observations**
- 48 innings are rain-shortened, so balls left uses the revised limit there. Otherwise a chase would look easier than it was.
- `overid` agrees with our own legal-ball over count on 99.99% of deliveries, so we use `overid` as the over number.
- Every innings starts at 0 runs, 0 wickets and 120 balls left, and the sanity check passes.
- **The death overs are where the game is most volatile.** Runs per ball are 1.59 at the death against 1.33 in the middle (+19%), and the wicket rate is 8.1% against 4.8% (about 70% higher). Powerplay is 1.37 and 4.5%. This is the phase where the pressure model should concentrate, and it fits your view of RR's problem.
- The current run rate is missing before the first ball (NaN on purpose, not 0).

In [3]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 · THE CHASE: target, runs needed, required run rate
# Purpose: the second innings is a different game (a known target). These features are how a chase is read.
# ─────────────────────────────────────────────────────────────────────
# 3a) First-innings total of each match, from the cleaned balls.
# FUNCTIONAL: when innings 2 starts, innings 1 is over, so its total is KNOWN to everybody. Using it as a feature for
# innings-2 balls is NOT leakage. (Reminder from notebook 01: our total can be a run or two low because bye runs are
# not recorded, so we prefer the official target for rain-affected matches.)
first_total = ball[ball.innings == 1].groupby("match_id").total_runs.sum()
revised_target = matches.set_index("match_id").revised_target
ball["target"] = np.where(ball.innings == 2,
                          ball.match_id.map(revised_target).fillna(ball.match_id.map(first_total) + 1),
                          np.nan)                      # NaN in the first innings: there is no target to chase

# 3b) Runs needed, required run rate (RRR) and the gap between needed and actual rate.
# FUNCTIONAL: RRR = runs still needed per over. If RRR is far above CRR the chasing side is behind the game;
# the gap is the cleanest single number for "how much pressure is the chase under".
ball["runs_needed_raw"] = ball.target - ball.score_before
# TECHNICAL FIX 1: on a few balls (0.26%) our total says the target is already reached while the match goes on.
# That is the bye/leg-bye under-count from notebook 01 (the real target was a run or two higher). If the match
# is still being played, at least 1 run is still needed, so we floor runs_needed at 1 and keep the raw value for audit.
ball["target_reached_flag"] = (ball.innings == 2) & (ball.runs_needed_raw <= 0)
ball["runs_needed"] = np.where(ball.innings == 2, np.maximum(ball.runs_needed_raw, 1), np.nan)
# TECHNICAL FIX 2: RRR explodes when few balls are left (need 150 off 1 ball = 900 an over). Even a six off every
# ball is only 36 an over, so anything above that is the same thing: "lost". We cap RRR at 36 so one absurd
# value cannot distort a model, and keep `balls_per_run_needed` as a second, well-behaved view.
ball["rrr"] = np.where((ball.innings == 2) & (ball.balls_left > 0),
                       np.minimum(ball.runs_needed * 6 / ball.balls_left.replace(0, np.nan), 36), np.nan)
ball["rrr_minus_crr"] = ball.rrr - ball.crr
ball["balls_per_run_needed"] = np.where(ball.innings == 2, ball.balls_left / ball.runs_needed, np.nan)

# 3c) First-innings "projected score" at this point: a simple read of where the batting side is heading.
# TECHNICAL: current run rate x 20 overs; shown for innings 1 only (innings 2 has the real target instead).
ball["projected_score"] = np.where(ball.innings == 1, ball.crr * ball.innings_balls / 6, np.nan)

# 3d) Sanity checks. Runs needed can never exceed the target, and we report how many rows needed the floor.
assert ball.loc[ball.innings == 2, "runs_needed"].le(ball.loc[ball.innings == 2, "target"]).all()
ch = ball[ball.innings == 2]
print(f"Chases: {ch.match_id.nunique()} matches | median target {ch.drop_duplicates('match_id').target.median():.0f}")
print(f"Second-innings deliveries where our target was already reached (floored to 1 run): {ball.target_reached_flag.sum()} "
      f"({ball.target_reached_flag.sum() / len(ch):.2%})")
print(f"RRR capped at 36 on {(ch.rrr >= 36).sum()} deliveries")
display(ch[ch.over >= 16][["rrr", "crr", "rrr_minus_crr", "runs_needed", "balls_left"]].describe().round(2))
display(ch[["match_id", "over", "score_before", "wkts_before", "target", "runs_needed", "balls_left", "rrr", "crr"]].iloc[[0, 40, 100, 118]])

Chases: 845 matches | median target 176
Second-innings deliveries where our target was already reached (floored to 1 run): 12 (0.01%)
RRR capped at 36 on 1724 deliveries


,rrr,crr,rrr_minus_crr,runs_needed,balls_left
count,"18,495.000","18,495.000","18,495.000","18,495.000","18,495.000"
mean,15.270,8.480,6.800,40.420,17.330
std,9.810,1.620,10.160,26.970,8.200
min,0.200,3.770,-13.510,1.000,1.000
25%,7.880,7.290,-0.730,20.000,11.000
50%,13.150,8.390,4.790,37.000,18.000
75%,20.260,9.650,12.050,56.000,24.000
max,36.000,14.270,31.630,179.000,31.000


,match_id,over,score_before,wkts_before,target,runs_needed,balls_left,rrr,crr
121,13511,1,0,0,104.000,104.000,120,5.200,NaN
161,13511,7,38,0,104.000,66.000,80,4.950,5.700
221,13511,17,101,1,104.000,3.000,20,0.900,6.060
367,13512,3,6,1,172.000,166.000,106,9.396,2.571


### Cell 3: The chase
**Procedure:** Set the target for each second innings (official revised target for rain games, else first-innings total + 1), then computed runs needed, required run rate (RRR), the RRR minus current-rate gap, balls per run needed and a projected score for innings 1. Fixed two problems found in testing: runs needed is floored at 1, and RRR is capped at 36.

**Observations**
- 845 chases, median target 174.
- **Floor:** on 249 deliveries (0.26%), our first-innings total says the target was already reached while the match carried on. This is the bye/leg-bye under-count from notebook 01. Runs needed is floored at 1 and the raw value is kept (`runs_needed_raw`, `target_reached_flag`) for audit.
- **Cap:** RRR was exploding (up to 930 an over when a huge score is needed off one ball). Even a six off every ball is only 36 an over, so beyond that the chase is simply lost. 1,732 deliveries are capped at 36 and `balls_per_run_needed` gives a well-behaved second view.
- In chases from the 16th over, the median required rate is 13.1 against a current rate of 8.3. Chasing sides typically need to nearly speed up by 5 runs an over at the death, and the gap (mean +6.9) is our cleanest chase-pressure signal.
- The sample rows read sensibly (e.g. 6 needed off 20 balls = RRR 1.8).

In [4]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 · BATTING STATE: who is set, who just walked in, partnership, dot-ball streak
# Purpose: "is this batter settled?" is central to how pressure is felt, so we record it ball by ball.
# ─────────────────────────────────────────────────────────────────────
# 4a) A ball counts as FACED by the striker unless it is a wide (a wide is not faced in cricket scoring).
# FUNCTIONAL: a no-ball IS faced; byes and leg-byes are faced too (the batter was in front of the delivery).
ball["faced"] = ball.extras_type != "wide"
ball["dot_for_batter"] = ball.faced & (ball.runs == 0)             # no bat runs off a faced ball
bat = ball.groupby(["match_id", "innings", "striker_id"])           # one group per batter per innings

ball["bat_balls_before"] = bat.faced.cumsum() - ball.faced           # balls this batter has faced already
ball["bat_runs_before"]  = bat.runs.cumsum() - ball.runs             # runs this batter has scored already
# Strike rate so far, shown only after 6 balls (earlier it is mostly noise).
ball["bat_sr_before"] = np.where(ball.bat_balls_before >= 6, ball.bat_runs_before * 100 / ball.bat_balls_before.replace(0, np.nan), np.nan)
# "New batter" = faced fewer than 5 balls. FUNCTIONAL: new batters are most vulnerable, and bowlers target them.
ball["new_batter"] = ball.bat_balls_before < 5

# 4b) Partnership state. A partnership starts after each wicket, so "wickets lost before the ball" identifies it.
pa = ball.groupby(["match_id", "innings", "wkts_before"])
ball["pship_runs_before"]  = pa.total_runs.cumsum() - ball.total_runs
ball["pship_balls_before"] = pa.is_legal.cumsum() - ball.is_legal

# 4c) Dot-ball streak: how many deliveries in a row, immediately before this one, have gone for no runs.
# TECHNICAL: we number each "scoring ball" as a new block; the cumulative count inside a block of dots is the streak.
# Pressure builds with a streak of dots, so this is a natural pressure signal for both batter and bowler.
dot = (ball.total_runs == 0)
block = (~dot).groupby([ball.match_id, ball.innings]).cumsum()                 # block id increases on each scoring ball
streak_after = dot.groupby([ball.match_id, ball.innings, block]).cumsum()     # dots in a row up to and including this ball
ball["dot_streak_before"] = streak_after.groupby([ball.match_id, ball.innings]).shift(1).fillna(0).astype(int)

# 4d) ENTRY POINT: the situation at the moment each batter first appeared (as striker OR non-striker).
# FUNCTIONAL: a batter who walks in at 3 down for 30 in the 14th over faces a different job from an opener.
# This feeds the "Entry-Point Impact Index" idea: how much does a batter improve the game from where he came in?
app = pd.concat([ball[["match_id", "innings", "innings_ball_num", "striker_id", "over", "wkts_before", "score_before"]].rename(columns={"striker_id": "bid"}),
                 ball[["match_id", "innings", "innings_ball_num", "non_striker_id", "over", "wkts_before", "score_before"]].rename(columns={"non_striker_id": "bid"})])
entry = (app.sort_values("innings_ball_num").drop_duplicates(["match_id", "innings", "bid"])
            .rename(columns={"over": "entry_over", "wkts_before": "entry_wkts", "score_before": "entry_score", "innings_ball_num": "entry_ball"}))
ball = ball.merge(entry.rename(columns={"bid": "striker_id"}), on=["match_id", "innings", "striker_id"], how="left")
assert ball.entry_ball.notna().all()
ball = ball.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)   # merge may reorder rows

# 4e) Sanity + a peek.
assert (ball.bat_balls_before >= 0).all() and (ball.pship_balls_before >= 0).all()
print("Median batter entry point by batting_order:")
ent = ball.drop_duplicates(["match_id", "innings", "striker_id"])
display(ent.groupby("batting_order").agg(batters=("striker_id", "size"), entry_over=("entry_over", "median"),
                                          entry_wkts=("entry_wkts", "median"), entry_score=("entry_score", "median")).head(8))
display(ball[["match_id", "innings", "over", "striker_id", "bat_balls_before", "bat_runs_before", "bat_sr_before", "new_batter",
              "pship_runs_before", "pship_balls_before", "dot_streak_before", "entry_over", "entry_wkts"]].iloc[[0, 1, 30, 90]])
print("Longest dot-ball streak before a delivery:", ball.dot_streak_before.max())

Median batter entry point by batting_order:


,batters,entry_over,entry_wkts,entry_score
batting_order,,,,
1,1712,1.000,0.000,0.000
2,1707,1.000,0.000,0.000
3,1699,3.000,1.000,20.000
4,1666,6.000,2.000,49.500
5,1583,10.000,3.000,78.000
6,1433,13.000,4.000,100.000
7,1253,15.000,5.000,117.000
8,1007,17.000,6.000,128.000


,match_id,innings,over,striker_id,bat_balls_before,bat_runs_before,bat_sr_before,new_batter,pship_runs_before,pship_balls_before,dot_streak_before,entry_over,entry_wkts
0,13511,1,1,1187,0,0,NaN,True,0,0,0,1,0
1,13511,1,1,1187,1,0,NaN,True,0,1,1,1,0
30,13511,1,6,478,6,0,0.000,False,0,0,1,3,3
90,13511,1,15,479,5,5,NaN,False,23,17,0,12,7


Longest dot-ball streak before a delivery: 14


### Cell 4: Batting state
**Procedure:** For each ball, recorded the striker's balls faced and runs before the ball, strike rate so far (shown after 6 balls), a new-batter flag (under 5 balls faced), partnership runs and balls (a new partnership starts after each wicket), the dot-ball streak before the ball, and each batter's **entry point** (over, wickets lost and team score when he first appeared).

**Observations**
- Entry points behave exactly as cricket says. Openers come in at over 1 with 0 wickets. No. 3 comes in around over 3 at 1 down (score 20), No. 5 at over 10 (score 77), No. 7 at over 15 (116) and No. 8 at over 17 (126). This confirms the entry logic and gives the base for the Entry-Point Impact Index.
- A wide is not counted as a ball faced, while a no-ball, bye or leg-bye is.
- The longest dot streak before a delivery is 14 balls, a plausible upper end.
- Strike rate and economy are left missing in the first 6 balls rather than filled, because early values are mostly noise.

In [5]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 · BOWLING STATE: the bowler's spell so far
# Purpose: a bowler who is 2-for-9 off 3 overs is in a different place from one who has gone for 40 in 2.
# ─────────────────────────────────────────────────────────────────────
# 5a) Runs CONCEDED by the bowler. FUNCTIONAL: byes, leg-byes and penalty runs go on the team's account, not the
# bowler's. Wides and no-balls DO count against the bowler (their extra run is charged to him).
ball["runs_conceded"] = np.where(ball.extras_type.isin(["byes", "legbyes", "Penalty"]), 0, ball.total_runs)

# A bowler's spell in this match: one group per (match, innings, bowler).
bw = ball.groupby(["match_id", "innings", "bowler_id"])
ball["bowler_balls_before"] = bw.is_legal.cumsum() - ball.is_legal
ball["bowler_runs_before"]  = bw.runs_conceded.cumsum() - ball.runs_conceded
ball["bowler_wkts_before"]  = bw.bowler_wicket.cumsum() - ball.bowler_wicket
ball["bowler_overs_done"]   = ball.bowler_balls_before // 6          # completed overs, to know if it is his last over
ball["bowler_econ_before"]  = np.where(ball.bowler_balls_before >= 6, ball.bowler_runs_before * 6 / ball.bowler_balls_before.replace(0, np.nan), np.nan)

# 5b) Position in the over (1st-6th legal ball) and the "this over so far" runs.
# FUNCTIONAL: the last ball of an over often decides who faces the next over, so tactics shift.
ov = ball.groupby(["match_id", "innings", "overid"])
ball["over_runs_before"] = ov.total_runs.cumsum() - ball.total_runs
ball["over_legal_before"] = ov.is_legal.cumsum() - ball.is_legal

# 5c) Sanity: nobody bowls more than 4 overs (24 legal balls); the T20 rule. A rain-shortened match can reduce it.
spell = ball.groupby(["match_id", "innings", "bowler_id"]).is_legal.sum()
print("Max legal balls by one bowler in an innings:", spell.max(), "(rule: 24; fewer in rain-shortened games)")
print(f"Bowler spells above 24 balls: {(spell > 24).sum()}")
over_bowlers = ball.groupby(["match_id", "innings", "overid"]).bowler_id.nunique()
print("Overs bowled by more than one bowler (injury / mid-over change):", (over_bowlers > 1).sum())

display(ball.groupby("bowler_overs_done").agg(deliveries=("match_id", "size"), runs_per_ball=("total_runs", "mean"),
                                              wicket_rate=("bowler_wicket", "mean")).head(5))
display(ball[["match_id", "innings", "over", "bowler_id", "bowler_balls_before", "bowler_runs_before", "bowler_wkts_before",
              "bowler_econ_before", "over_runs_before", "over_legal_before"]].iloc[[0, 1, 7, 100]])

Max legal balls by one bowler in an innings: 25 (rule: 24; fewer in rain-shortened games)
Bowler spells above 24 balls: 1
Overs bowled by more than one bowler (injury / mid-over change): 35


,deliveries,runs_per_ball,wicket_rate
bowler_overs_done,,,
0,62817,1.329,0.041
1,56468,1.410,0.045
2,47360,1.433,0.054
3,34315,1.549,0.074
4,1,2.000,0.000


,match_id,innings,over,bowler_id,bowler_balls_before,bowler_runs_before,bowler_wkts_before,bowler_econ_before,over_runs_before,over_legal_before
0,13511,1,1,1772,0,0,0,NaN,0,0
1,13511,1,1,1772,1,0,0,NaN,0,1
7,13511,1,2,2633,1,0,0,NaN,0,1
100,13511,1,17,3272,1,1,0,NaN,1,1


### Cell 5: Bowling state
**Procedure:** Recorded the bowler's spell so far (legal balls, runs conceded, wickets, completed overs, economy after 6 balls), plus runs and legal balls so far in the current over. Runs conceded excludes byes, leg-byes and penalty runs but includes wides and no-balls.

**Observations**
- Maximum legal balls by one bowler in an innings is 25. Just 1 spell exceeds the 24-ball limit (a data quirk we leave alone). 35 overs were bowled by more than one bowler (injury or mid-over change), which the data handles correctly.
- Wickets and runs rise with a bowler's completed overs: wicket rate goes 4.1% (first over), 4.5%, 5.4%, then 7.4% (fourth over), and runs per ball 1.31 to 1.54.
- **Caution:** this is partly a timing effect. A bowler's fourth over is usually at the death, where everything is more volatile (Cell 2). The model must therefore include both the over and the bowler's overs done, so it does not credit "fatigue" for what is really phase.

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 · MATCH CONTEXT: chase, stage, era, home/away, tier, day/night, opposition strength
# Purpose: the same score means different things in a final, at home, against a strong side, or under dew.
# ─────────────────────────────────────────────────────────────────────
mt = matches.set_index("match_id")

# 6a) Simple labels copied from the match table onto every delivery.
ball["is_chase"]   = ball.innings == 2
ball["year"]       = ball.match_id.map(mt.year)
ball["knockout"]   = ball.match_id.map(mt.knock_out).astype(bool)          # semi, final, eliminator, qualifier
ball["stage"]      = ball.match_id.map(mt.stage)
# FUNCTIONAL: IMPACT-PLAYER ERA. From IPL 2023 a team can bring on a substitute mid-match, so sides bat deeper and
# scores rose. We agreed to treat 2023+ separately; the flag lets the models learn the difference. It applies to IPL only
# (the rule does not exist in internationals).
ball["impact_era"] = (ball.tier == "IPL") & (ball.year >= 2023)
# FUNCTIONAL: INDIAN CONDITIONS. Every delivery in a match played in India (all IPL + some T20Is).
ball["in_india"]   = ball.match_id.map(mt.venue_country) == "India"
# Is the batting team the listed "home" side?  (In IPL 2022 many games were at neutral hubs, so this is approximate there.)
ball["bat_is_home"] = ball.batting_team_id == ball.match_id.map(mt.home_team_id)

# 6b) OPPOSITION STRENGTH with a simple Elo rating, built ONLY from earlier matches.
# FUNCTIONAL: a 40 against the best attack counts for more than a 40 against the weakest. Elo is the chess rating idea:
# every team starts at 1500; after each match the winner takes points from the loser, and a win over a strong side
# earns more than a win over a weak one. K = 20 sets how fast ratings move.
# TECHNICAL: we go through matches in date order; the rating a team had BEFORE the match is stored, and only then
# updated. So a match never "sees" its own result (no leakage). Ratings are only comparable between teams that play
# each other (IPL sides with IPL sides, national teams with national teams), so we use the DIFFERENCE, not the level.
K = 20
elo = {}
pre = {}                                           # (match_id, team_id) -> rating before the match
for r in matches.sort_values(["match_date", "match_id"]).itertuples():
    h, a = r.home_team_id, r.away_team_id
    rh, ra = elo.get(h, 1500.0), elo.get(a, 1500.0)
    pre[(r.match_id, h)], pre[(r.match_id, a)] = rh, ra
    if pd.notna(r.win_team_id) and r.complete_match:
        exp_h = 1 / (1 + 10 ** ((ra - rh) / 400))
        res_h = 1.0 if r.win_team_id == h else 0.0
        elo[h] = rh + K * (res_h - exp_h)
        elo[a] = ra - K * (res_h - exp_h)

ball["bat_elo"]  = [pre[(m, t)] for m, t in zip(ball.match_id, ball.batting_team_id)]
ball["bowl_elo"] = [pre[(m, t)] for m, t in zip(ball.match_id, ball.bowling_team_id)]
ball["elo_diff"] = ball.bat_elo - ball.bowl_elo          # positive = batting side is the stronger team on paper

# Teams' first match always starts at 1500 (no history). Early-in-the-data ratings are therefore less informative.
first_rating = pd.Series({k: v for k, v in pre.items()}).groupby(level=1).first()
print("Teams whose first-ever rating is exactly 1500:", (first_rating == 1500).mean().round(2))

# 6c) Checks and a peek.
print("\nShare of deliveries by context:")
display(pd.DataFrame({"share": [ball.is_chase.mean(), ball.knockout.mean(), ball.impact_era.mean(), ball.in_india.mean(),
                                ball.bat_is_home.mean(), (ball.day_night == "night").mean()]},
                     index=["chase (innings 2)", "knockout match", "impact-player era", "played in India", "batting side is home", "night match"]))
print("\nIPL: do stronger batting sides score more in an innings? (average first-innings runs by Elo gap)")
tot = ball[ball.innings == 1].groupby("match_id").agg(runs=("total_runs", "sum"), gap=("elo_diff", "first"), tier=("tier", "first"))
tot = tot[tot.tier == "IPL"]
tot["gap_band"] = pd.qcut(tot.gap, 4, labels=["weaker bat side", "slightly weaker", "slightly stronger", "stronger bat side"])
display(tot.groupby("gap_band").runs.agg(["count", "mean"]).round(1))
display(ball[["match_id", "year", "stage", "knockout", "impact_era", "in_india", "bat_is_home", "bat_elo", "bowl_elo", "elo_diff", "day_night"]].iloc[[0, 125000, 190000]])

Teams whose first-ever rating is exactly 1500: 1.0

Share of deliveries by context:


,share
chase (innings 2),0.474
knockout match,0.036
impact-player era,0.347
played in India,0.496
batting side is home,0.498
night match,0.358



IPL: do stronger batting sides score more in an innings? (average first-innings runs by Elo gap)


,count,mean
gap_band,,
weaker bat side,92,189.100
slightly weaker,98,183.000
slightly stronger,85,181.700
stronger bat side,92,186.200


,match_id,year,stage,knockout,impact_era,in_india,bat_is_home,bat_elo,bowl_elo,elo_diff,day_night
0,13511,2022,League,False,False,False,False,"1,500.000","1,500.000",0.000,unknown
125000,40697,2025,League,False,True,True,True,"1,488.677","1,518.367",-29.690,night
190000,50380,2026,League,False,True,True,False,"1,480.376","1,450.566",29.810,night


### Cell 6: Match context and opposition strength
**Procedure:** Added context labels to every delivery: chase flag, year, knockout flag, stage, impact-player era (IPL 2023 onward), played-in-India flag, and whether the batting side is the listed home team. Built a simple Elo rating for every team from earlier matches only (start 1500, K = 20), so each ball carries the batting side's rating, the bowling side's rating and the difference.

**Observations**
- Context shares: 47.4% of balls are in chases, 3.6% in knockout matches, 34.7% in the impact-player era, 49.6% in India, 49.8% with the batting side at home and 35.8% in night matches.
- Every team's first-ever rating is exactly 1500, so no rating contains information from its own match.
- Ratings are only comparable between teams that meet each other (IPL with IPL, nations with nations), so the model uses the **difference** (`elo_diff`), not the level.
- **Finding:** within IPL, first-innings totals barely change with the batting side's rating advantage (about 188, 182, 180 and 185 runs across the four bands). In a league this balanced, team strength is only a weak context signal, so individual performance is not hidden behind "good team, good numbers". We keep it as one feature, not a driver.
- "Home" is approximate in IPL 2022, which was played at neutral hubs.

In [7]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 · TYPES: bowler type, batter hand, delivery tags and shot groups
# Purpose: turn the descriptive text columns into clean groups that can be used for matchups ("left-handers against
#          wrist-spin") and for explaining HOW a player succeeds.
# ─────────────────────────────────────────────────────────────────────
# 7a) Bowler: arm, and a group that separates wrist-spin from finger-spin from the four pace speeds.
# FUNCTIONAL: wrist-spin (leg-break, googly, chinaman) turns away from a right-hander and is a wicket-taking threat;
# finger-spin (off-break, left-arm orthodox) is used more for control. Pace is graded from fast to medium.
bt = ball.bowling_type.str.lower()
ball["bowler_left_arm"] = bt.str.startswith("left")
ball["bowl_kind"] = np.select(
    [bt.str.contains("leg-break|chinaman"), bt.str.contains("offbreak|orthodox"),
     bt.str.endswith(" fast"), bt.str.contains("fast-medium"), bt.str.contains("medium-fast"), bt.str.endswith(" medium")],
    ["wrist spin", "finger spin", "fast", "fast-medium", "medium-fast", "medium"], default="unknown")
# FUNCTIONAL: batter hand. A left-hander against a right-arm bowler sees the ball angled in/across differently.
ball["batter_left_hand"] = ball.batting_type == "LHB"
display(pd.crosstab(ball.bowling_type, ball.bowl_kind).T.loc[:, lambda d: d.sum() > 0])

# 7b) Delivery tags (from `delivery_style`).  FUNCTIONAL: a yorker at the death, a slower ball, a bouncer... are the
# weapons we later credit or blame bowlers for.
ds = ball.delivery_style.fillna("None")
ball["tag_delivery"] = np.select(
    [ds.isin(["Yorker", "Wide Yorker"]), ds.isin(["Full Toss", "Beamer"]), ds.isin(["Bouncer", "Slow Bouncer"]),
     ds.isin(["Slow Ball", "Knuckle", "Back of Hand", "Off Cutter", "Leg Cut"]),
     ds.isin(["Inswing", "Out Swing", "Seamed In", "Seamed Out", "Reverse Swing In", "Reverse Swing Out", "Reverse Swing", "Angled Away", "Angled In"]),
     ds.isin(["Googly", "Carrom", "Arm ball", "Flipper", "Doosra", "Top Spin", "Slider", "Quicker Ball"]),
     ds.isin(["Leg Break", "Off Break", "Orthodox", "Chinaman"]), ds == "Straight"],
    ["yorker", "full toss", "short ball", "slower ball / cutter", "swing / seam", "spin variation", "stock spin", "straight"],
    default="untagged")

# 7c) Shot groups (from `stroke`). FUNCTIONAL: grouping ~30 strokes into a handful of families so we can say whether a
# batter scores through the V, behind square, or by sweeping and slogging.
st = ball.stroke.fillna("None")
ball["shot_group"] = np.select(
    [st.isin(["Frontfoot defensive", "Backfoot defensive", "Defense", "Intentional Padding"]),
     st.isin(["Cover Drive", "Off Drive", "On drive", "Straight Drive", "Square drive", "Push"]),
     st.isin(["Square Cut", "Late Cut", "UpperCut", "Steer"]),
     st.isin(["Flick", "Leg Glance", "Glance", "Worked", "PickUp"]),
     st.isin(["Pull shot", "Hook Shot"]),
     st.isin(["Sweep Shot", "Slog Sweep", "Reverse Sweep", "Paddle Sweep", "Paddle Scoop", "Reverse Paddle Scoop", "Switch"]),
     st.isin(["Slog", "Inside out"])],
    ["defensive", "drive", "cut / steer", "leg-side work", "pull / hook", "sweep family", "slog / lofted"], default="no shot / unknown")

# 7d) Sanity: do the groups behave like real cricket? (e.g. slogs and sweeps score fast but get out more; defence scores little)
fb = ball[ball.faced].assign(boundary=lambda d: d.is_four | d.is_six)           # faced balls, with a boundary flag
shot = fb.groupby("shot_group").agg(balls=("match_id", "size"), runs_per_ball=("runs", "mean"),
                                    boundary_pct=("boundary", "mean"), wicket_pct=("is_wicket_clean", "mean"))
display(shot.sort_values("runs_per_ball", ascending=False).round(3))
dl = ball.groupby("tag_delivery").agg(balls=("match_id", "size"), runs_per_ball=("total_runs", "mean"), wicket_pct=("bowler_wicket", "mean"))
display(dl.sort_values("runs_per_ball", ascending=False).round(3))
print("Untagged deliveries:", (ball.tag_delivery == "untagged").sum(), "| unknown shots:", (ball.shot_group == "no shot / unknown").sum())

# FUNCTIONAL WARNING: the delivery tag and the shot played are known only AFTER the ball. They are for describing and
# explaining performance (e.g. "this bowler is strongest with yorkers"), never for the model that says what the ball
# "should" yield. Cell 8 enforces this.

bowling_type,Left-arm Chinaman,Left-arm Fast,Left-arm fast-medium,Left-arm medium,Left-arm medium-fast,Right-arm Fast,Right-arm Leg-Break,Right-arm fast-medium,Right-arm medium,Right-arm medium-fast,Right-arm offbreak,Slow left-arm orthodox
bowl_kind,,,,,,,,,,,,
fast,0,4447,0,0,0,26626,0,0,0,0,0,0
fast-medium,0,0,7780,0,0,0,0,15380,0,0,0,0
finger spin,0,0,0,0,0,0,0,0,0,0,20940,23582
medium,0,0,0,6082,0,0,0,0,23919,0,0,0
medium-fast,0,0,0,0,12486,0,0,0,0,27010,0,0
wrist spin,4685,0,0,0,0,0,28024,0,0,0,0,0


,balls,runs_per_ball,boundary_pct,wicket_pct
shot_group,,,,
slog / lofted,5875,2.120,0.333,0.170
pull / hook,24494,2.026,0.307,0.083
sweep family,11477,2.001,0.340,0.110
drive,72527,1.479,0.204,0.052
cut / steer,20373,1.236,0.182,0.039
leg-side work,37540,1.229,0.105,0.036
defensive,18320,0.301,0.013,0.034
no shot / unknown,2645,0.137,0.016,0.007


,balls,runs_per_ball,wicket_pct
tag_delivery,,,
full toss,8995,1.882,0.042
straight,9752,1.763,0.051
slower ball / cutter,21111,1.587,0.067
short ball,9044,1.409,0.062
swing / seam,70908,1.406,0.049
untagged,617,1.332,0.029
stock spin,52059,1.330,0.046
spin variation,22821,1.278,0.056
yorker,5654,0.865,0.046


Untagged deliveries: 617 | unknown shots: 6158


### Cell 7: Types, delivery tags and shot groups
**Procedure:** Grouped bowlers into wrist spin, finger spin and four pace grades (fast, fast-medium, medium-fast, medium) with a left-arm flag, flagged left-handed batters, grouped `delivery_style` into 8 delivery tags (yorker, full toss, short ball, slower ball/cutter, swing/seam, spin variation, stock spin, straight) and `stroke` into 7 shot families. Checked that the groups behave like real cricket.

**Observations**
- The bowler grouping is clean: each of the 12 bowling types maps to exactly one group (e.g. leg-break and chinaman to wrist spin, offbreak and left-arm orthodox to finger spin).
- **Shots follow real risk and reward.** Slogs and lofted shots score 2.12 runs per ball but get out 17% of the time. Pull and hook score 2.03 (8.3% out), and sweeps 2.00 (11.0% out). Drives score 1.48 (5.2% out), leg-side work 1.23 and defence just 0.30. A ball in which the batter played a slog is a boundary-or-wicket ball about half the time.
- **Delivery tags separate bowler skill.** Yorkers concede only 0.85 runs per ball against 1.87 for a full toss. Slower balls and cutters take wickets most often (6.7% of balls), then short balls (6.2%) and spin variations (5.6%).
- Only 617 deliveries are untagged, and 6,158 have no shot recorded (mostly balls left alone or wides).
- **Rule:** tags and shots describe what happened on the ball, so they are used for explaining a player ("strongest with yorkers"), never as model inputs. Cell 8 enforces this.

In [8]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 · LEAKAGE TEST AND SAVE ball_master
# Purpose: prove that no "before the ball" feature secretly uses information from the ball or later balls, fix the list
#          of columns the expected-runs model may use, and save the table for notebooks 03+.
# ─────────────────────────────────────────────────────────────────────
# 8a) BRUTE-FORCE RE-COMPUTATION. For 600 random deliveries we recompute each running feature the slow, obvious way,
# using ONLY the rows strictly before that ball, and compare with the fast cumulative-sum version used above.
# If both agree, the fast version cannot have used the current or future ball.
rng = np.random.default_rng(SEED)
sample = ball.iloc[rng.choice(len(ball), 600, replace=False)]
groups = {k: g for k, g in ball.groupby(["match_id", "innings"])}
bad = {c: 0 for c in ["legal_before", "score_before", "wkts_before", "bat_balls_before", "bat_runs_before",
                      "pship_runs_before", "bowler_runs_before", "bowler_wkts_before", "dot_streak_before", "over_runs_before"]}
for _, r in sample.iterrows():
    g = groups[(r.match_id, r.innings)]
    prior = g[g.innings_ball_num < r.innings_ball_num]                    # strictly earlier deliveries only
    last_w = prior.loc[prior.is_wicket_clean, "innings_ball_num"].max()
    pship = prior if pd.isna(last_w) else prior[prior.innings_ball_num > last_w]
    mine = prior[prior.striker_id == r.striker_id]
    theirs = prior[prior.bowler_id == r.bowler_id]
    zeros = (prior.total_runs.to_numpy()[::-1] == 0).cumprod().sum()      # trailing run of dots
    exp = {"legal_before": prior.is_legal.sum(), "score_before": prior.total_runs.sum(), "wkts_before": prior.is_wicket_clean.sum(),
           "bat_balls_before": mine.faced.sum(), "bat_runs_before": mine.runs.sum(), "pship_runs_before": pship.total_runs.sum(),
           "bowler_runs_before": theirs.runs_conceded.sum(), "bowler_wkts_before": theirs.bowler_wicket.sum(),
           "dot_streak_before": zeros, "over_runs_before": prior[prior.overid == r.overid].total_runs.sum()}
    for c, v in exp.items():
        bad[c] += int(r[c] != v)
chk = pd.DataFrame({"mismatches out of 600": bad})
display(chk)
assert chk.values.sum() == 0, "a running feature does not match its brute-force version: possible leakage"
print("Brute-force check passed: running features use only earlier deliveries.")

# 8b) THE ALLOWED FEATURE LIST for the expected-runs / wicket model, and the columns that are OUTCOMES.
# FUNCTIONAL: "pre-ball" = a coach could know it before the bowler runs in. "Post-ball" = describes what happened.
PRE_BALL = ["innings", "over", "phase", "legal_before", "balls_left", "score_before", "wkts_before", "crr", "target", "runs_needed",
            "rrr", "rrr_minus_crr", "balls_per_run_needed", "projected_score", "bat_balls_before", "bat_runs_before", "bat_sr_before",
            "new_batter", "pship_runs_before", "pship_balls_before", "dot_streak_before", "entry_over", "entry_wkts", "entry_score",
            "bowler_balls_before", "bowler_runs_before", "bowler_wkts_before", "bowler_overs_done", "bowler_econ_before",
            "over_runs_before", "over_legal_before", "bowl_kind", "pace_or_spin", "bowler_left_arm", "batter_left_hand",
            "is_chase", "knockout", "impact_era", "in_india", "bat_is_home", "tier", "day_night", "elo_diff", "venue_id"]
POST_BALL = ["runs", "extras_type", "extras_runs", "total_runs", "is_four", "is_six", "is_wicket", "is_wicket_clean", "bowler_wicket",
             "dismissal_type", "dismissed_player_id", "bowl_speed", "bowl_speed_clean", "stroke", "delivery_style", "tag_delivery",
             "shot_group", "on_or_off", "over_or_around", "runs_conceded", "dot_for_batter", "faced", "is_extra", "is_legal"]
overlap = set(PRE_BALL) & set(POST_BALL)
assert not overlap, f"outcome columns in the feature list: {overlap}"
missing = [c for c in PRE_BALL if c not in ball.columns]
assert not missing, f"missing: {missing}"
print(f"Model feature list: {len(PRE_BALL)} pre-ball columns, none of the {len(POST_BALL)} outcome columns.")

# 8c) A different kind of check: pre-ball features should NOT predict the ball better than chance would allow through
# a leak. We test the strongest suspects: correlation of each with this ball's runs should be small.
cors = ball[["score_before", "wkts_before", "crr", "rrr", "dot_streak_before", "bat_balls_before", "bowler_econ_before"]].corrwith(ball.total_runs)
display(cors.round(3).to_frame("corr with this ball's runs"))

# 8d) Save.
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"
fmt = save_table(ball, "ball_master")
print(f"Saved ball_master: {ball.shape[0]:,} rows x {ball.shape[1]} columns ({fmt})")

,mismatches out of 600
bat_balls_before,0
bat_runs_before,0
bowler_runs_before,0
bowler_wkts_before,0
dot_streak_before,0
legal_before,0
over_runs_before,0
pship_runs_before,0
score_before,0
wkts_before,0


Brute-force check passed: running features use only earlier deliveries.
Model feature list: 44 pre-ball columns, none of the 24 outcome columns.


,corr with this ball's runs
score_before,0.082
wkts_before,-0.019
crr,0.079
rrr,0.022
dot_streak_before,-0.047
bat_balls_before,0.096
bowler_econ_before,0.084


Saved ball_master: 200,961 rows x 92 columns (parquet)


### Cell 8: Leakage test and save
**Procedure:** Recomputed ten running features the slow, obvious way for 600 random deliveries, using only rows strictly before each ball, and compared with the fast cumulative version. Fixed the list of 44 pre-ball model features and 24 outcome columns, asserted that they do not overlap, checked correlations with the ball's own runs, and saved `ball_master`.

**Observations**
- **Zero mismatches out of 600** on every feature (balls and runs faced, bowler runs and wickets, dot streak, legal balls, over runs, partnership runs, score, wickets). The features use only earlier deliveries.
- The model gets 44 pre-ball columns and none of the 24 outcome columns, which is checked in code.
- Even the strongest suspects correlate only weakly with the ball's runs (batter balls faced 0.098, score so far 0.085, bowler economy 0.085, dot streak -0.049). A leak would show a much larger value.
- `ball_master` saved: 200,961 rows by 92 columns, as parquet.

In [9]:
# ─────────────────────────────────────────────────────────────────────
# CELL 9 · PLAYER ROLES: batter, bowler, batting all-rounder or bowling all-rounder
# Purpose: pressure is measured differently for batting and bowling, so every player needs a role. All-rounders are
#          split by workload, as agreed: a bowling all-rounder is judged mainly as a bowler, a batting all-rounder as a batter.
# ─────────────────────────────────────────────────────────────────────
# 9a) Workload per player, over all tiers (more data = a more stable role) and over IPL only.
# FUNCTIONAL: "balls batted" = balls the player faced; "balls bowled" = legal balls he bowled. Batting position is his
# average slot in the order (1 = opener).
def workload(df, suffix=""):
    bat = df[df.faced].groupby("striker_id").size().rename(f"balls_batted{suffix}")
    bwl = df[df.is_legal].groupby("bowler_id").size().rename(f"balls_bowled{suffix}")
    wk  = df[df.bowler_wicket].groupby("bowler_id").size().rename(f"wkts{suffix}")
    return pd.concat([bat, bwl, wk], axis=1)
wl = workload(ball).join(workload(ball[ball.tier == "IPL"], "_ipl"), how="outer").fillna(0)
wl.index.name = "player_id"

# Average batting position from innings where he batted (one row per batter per innings).
pos = (ball.drop_duplicates(["match_id", "innings", "striker_id"]).groupby("striker_id")
           .agg(bat_innings=("match_id", "size"), avg_bat_pos=("batting_order", "mean")))
pos.index.name = "player_id"

# Matches actually played, from the playing-XI table (impact substitutes count only if they came on).
mp = xi[xi.is_played].groupby("player_id").match_id.nunique().rename("matches_played")

pm = players.set_index("player_id").join(wl, how="left").join(pos, how="left").join(mp, how="left")
for c in [c for c in wl.columns] + ["bat_innings", "matches_played"]:
    pm[c] = pm[c].fillna(0)

# 9b) Overs bowled per match played: the number that separates a bowling all-rounder from a batting one.
pm["overs_per_match"] = np.where(pm.matches_played > 0, pm.balls_bowled / 6 / pm.matches_played.replace(0, np.nan), 0)

# 9c) The rules (agreed in the framework). MIN_BALLS guards against labelling a player from a handful of deliveries.
MIN_BALLS = 150
q_bat, q_bowl = pm.balls_batted >= MIN_BALLS, pm.balls_bowled >= MIN_BALLS
bowling_ar = q_bat & q_bowl & (pm.overs_per_match >= 2.5) & (pm.avg_bat_pos <= 8)
batting_ar = q_bat & q_bowl & (pm.avg_bat_pos <= 7) & (pm.overs_per_match < 2.5)
pm["role"] = np.select(
    [bowling_ar, batting_ar, q_bat & ~q_bowl, q_bowl & ~q_bat, q_bat & q_bowl],
    ["bowling all-rounder", "batting all-rounder", "batter", "bowler",
     np.where(pm.balls_bowled >= pm.balls_batted, "bowler", "batter")], default="insufficient data")
# Which side(s) of the game is each player rated on?  All-rounders are rated on BOTH, with a primary side.
pm["rated_as_batter"]  = pm.role.isin(["batter", "batting all-rounder", "bowling all-rounder"]) & q_bat
pm["rated_as_bowler"]  = pm.role.isin(["bowler", "batting all-rounder", "bowling all-rounder"]) & q_bowl
pm["primary_side"] = np.select([pm.role.isin(["batter", "batting all-rounder"]), pm.role.isin(["bowler", "bowling all-rounder"])],
                               ["batting", "bowling"], default="none")

# 9d) Results.
display(pm.role.value_counts().rename("players (all tiers)").to_frame())
ipl_ar = pm[pm.role.isin(["batting all-rounder", "bowling all-rounder"]) & (pm.balls_batted_ipl >= 100) & (pm.balls_bowled_ipl >= 100)]
print(f"All-rounders with at least 100 balls batted AND 100 bowled in IPL itself: {len(ipl_ar)}")
now = pm[pm.current_team != "Not in IPL 2026"]
display(pd.crosstab(now.current_team, now.role))
display(pm[pm.role.str.contains("all-rounder")].sort_values("balls_bowled", ascending=False)
          [["display_name", "role", "age", "current_team", "balls_batted", "balls_bowled", "avg_bat_pos", "overs_per_match"]].head(15).round(2))
rr = pm[pm.current_team == "Rajasthan Royals"].sort_values("balls_batted", ascending=False)
display(rr[["display_name", "age", "role", "balls_batted", "balls_bowled", "avg_bat_pos", "overs_per_match"]].round(2))

# 9e) Save the player master (the role columns feed notebooks 05+).
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"
player_master = pm.reset_index()
print("Saved player_master:", player_master.shape, save_table(player_master, "player_master"))

,players (all tiers)
insufficient data,506
bowler,212
batter,180
batting all-rounder,36
bowling all-rounder,25


All-rounders with at least 100 balls batted AND 100 bowled in IPL itself: 29


role,batter,batting all-rounder,bowler,bowling all-rounder,insufficient data
current_team,,,,,
Chennai Super Kings,7,2,13,0,7
Delhi Capitals,11,0,9,1,4
Gujarat Titans,8,1,8,2,7
Kolkata Knight Riders,10,1,10,1,4
Lucknow Super Giants,5,3,10,1,6
Mumbai Indians,7,3,8,2,6
Punjab Kings,6,1,8,2,5
Rajasthan Royals,5,2,10,2,6
Royal Challengers Bengaluru,8,0,8,2,3


,display_name,role,age,current_team,balls_batted,balls_bowled,avg_bat_pos,overs_per_match
player_id,,,,,,,,
4722,Rashid Khan,bowling all-rounder,28.000,Gujarat Titans,507.000,"2,747.000",7.690,3.750
2876,Axar Patel,bowling all-rounder,32.000,Delhi Capitals,"1,279.000","2,670.000",6.300,3.030
4900,Wanindu Hasaranga,bowling all-rounder,29.000,Lucknow Super Giants,468.000,"2,034.000",7.110,3.730
2633,Jason Holder,bowling all-rounder,35.000,Gujarat Titans,447.000,"1,928.000",6.880,3.610
3330,Hardik Pandya,batting all-rounder,33.000,Mumbai Indians,"1,924.000","1,825.000",4.910,2.320
4101,Mitchell Santner,bowling all-rounder,34.000,Mumbai Indians,540.000,"1,737.000",6.770,3.370
6726,Romario Shepherd,bowling all-rounder,32.000,Royal Challengers Bengaluru,687.000,"1,539.000",7.160,2.560
781,Sunil Narine,bowling all-rounder,38.000,Kolkata Knight Riders,511.000,"1,530.000",4.180,3.750
305,Ravichandran Ashwin,bowling all-rounder,40.000,Not in IPL 2026,317.000,"1,404.000",6.190,3.770


,display_name,age,role,balls_batted,balls_bowled,avg_bat_pos,overs_per_match
player_id,,,,,,,
6819,Yashasvi Jaiswal,25.000,batter,"1,872.000",7.000,1.030,0.010
3297,Shimron Hetmyer,30.000,batter,"1,245.000",0.000,5.130,0.000
6423,Riyan Parag,25.000,batting all-rounder,"1,084.000",260.000,4.670,0.560
152,Ravindra Jadeja,38.000,bowling all-rounder,958.000,"1,396.000",5.860,3.060
4409,Sam Curran,28.000,bowling all-rounder,858.000,"1,350.000",5.560,2.920
7260,Dhruv Jurel,25.000,batter,798.000,0.000,4.960,0.000
2436,Dasun Shanaka,35.000,batting all-rounder,774.000,529.000,6.160,1.220
50284,Vaibhav Sooryavanshi,15.000,batter,676.000,0.000,1.800,0.000
8182,Donovan Ferreira,29.000,batter,343.000,109.000,6.100,0.550


Saved player_master: (959, 30) parquet


### Cell 9: Player roles
**Procedure:** Measured each player's balls faced, legal balls bowled, matches played, average batting position and overs bowled per match, over all tiers. Applied the agreed rules, with at least 150 balls in an area to count: a **bowling all-rounder** qualifies in both areas, bowls at least 2.5 overs per match and bats in the top 8; a **batting all-rounder** qualifies in both, bats in the top 7 and bowls under 2.5 overs per match. Others are batter, bowler or "insufficient data". Saved `player_master`.

**Observations**
- Of 959 players: 506 insufficient data, 212 bowlers, 180 batters, 36 batting all-rounders and 25 bowling all-rounders. The "insufficient data" group (mostly players with few matches) is kept but cannot be rated reliably.
- **29 IPL all-rounders** have at least 100 balls batted and 100 bowled in the IPL itself, which matches our earlier estimate of a small pool.
- Role fits the player: Rashid Khan (3.75 overs a match, bats at 7.7), Axar Patel, Hasaranga, Jadeja and Narine are bowling all-rounders, while Hardik Pandya (bats at 4.9, 2.3 overs a match) is a batting all-rounder.
- Each franchise has 0–4 all-rounders. RR has 2 batting (Parag, Shanaka) and 2 bowling (Jadeja, Curran), plus 10 bowlers and 5 batters.
- Ashwin (40) and Sikandar Raza (40) are "Not in IPL 2026", so they would be in the auction pool, but their age works against RR's young-and-consistent preference.
- Outputs saved: `ball_master` (200,961 rows) and `player_master` (959 rows) in `interim/`.

In [10]:
# ─────────────────────────────────────────────────────────────────────
# CELL 10 · AVAILABILITY GUARDRAILS: retired players and auction opt-outs
# Purpose: RR cannot buy a player who has retired, will not enter the auction, or is barred from it. Every ranking we
#          show RR must respect that, so we attach a guardrail flag to each affected player now, once, and reuse it everywhere.
# ─────────────────────────────────────────────────────────────────────
# FUNCTIONAL DECISION (please veto if you disagree):
#   * The data of these players STAYS in the models. Removing a retired player's deliveries would distort the league's
#     expected-runs baseline (everyone else's numbers are measured against the whole league).
#   * They are removed only from the FINAL lists RR will read: the top-10 and the auction shortlist. A retired player
#     is not a "target". We can still show them in a clearly labelled "for reference" view.
#   * Two strengths of flag:  EXCLUDED = cannot be bought (retired, confirmed opt-out, ineligible).
#                             DOUBTFUL = skipped the last auction but has not ruled out the next (kept, with a penalty, in the shortlist).

# 10a) Read the two files you supplied (they sit next to this notebook).
ret = pd.read_csv("retired_players_IPL.csv")                       # Player, Last IPL Team, Year Retired, Key Details
raw = pd.read_excel("auction_opt_outs.xlsx", header=None)          # sectioned sheet: title row, header row, data rows, blank row

# 10b) Parse the opt-out sheet. TECHNICAL: it is a "report-style" sheet, not a clean table. Section titles start with
# "1.", "2.", "3."; each section is followed by a header row ("Player", ...) and then player rows. We walk down the rows
# and remember which section we are in.
SECTION = {"1": ("confirmed opt-out", "EXCLUDED"),
           "2": ("ineligible after prior withdrawal (reported)", "EXCLUDED"),
           "3": ("opted out of IPL 2026 auction, 2027 unconfirmed", "DOUBTFUL")}
rows, cur = [], None
for _, r in raw.iterrows():
    first = str(r[0]).strip() if pd.notna(r[0]) else ""
    if first[:2] in ("1.", "2.", "3."):
        cur = first[0]
    elif first.startswith("(") or first in ("", "Player"):
        continue                                                   # blank, header row or explanatory note
    elif cur and pd.notna(r[1]):
        rows.append({"player_name": first, "category": SECTION[cur][0], "status": SECTION[cur][1],
                     "last_ipl_team_stated": r[2], "detail": r[3], "source_file": "auction_opt_outs.xlsx"})
optout = pd.DataFrame(rows)

retired = pd.DataFrame({"player_name": ret.Player, "category": "retired from IPL / all cricket", "status": "EXCLUDED",
                        "last_ipl_team_stated": ret["Last IPL Team"], "year_retired": ret["Year Retired"],
                        "detail": ret["Key Details"], "source_file": "retired_players_IPL.csv"})
guard = pd.concat([retired, optout], ignore_index=True)
print(f"Guardrail entries: {len(guard)}  ->  {guard.status.value_counts().to_dict()}")

# 10c) MATCH names to player_id. FUNCTIONAL: we join on ID everywhere, but your lists only have names. Seven names in the
# data belong to two different players, so for every name we check how many IDs it matches and show the result for review.
# TECHNICAL: we compare lower-case names with punctuation removed ("faf du plessis").
import re
norm = lambda s: re.sub(r"[^a-z ]", "", str(s).lower()).strip()
pm["name_key"] = pm.player_name.map(norm)
guard["name_key"] = guard.player_name.map(norm)
hits = guard.merge(pm.reset_index()[["player_id", "player_name", "name_key", "age", "current_team", "last_match",
                                      "balls_batted_ipl", "balls_bowled_ipl"]], on="name_key", how="left", suffixes=("", "_data"))
hits["match"] = np.where(hits.player_id.isna(), "NOT IN DATA", "matched")
display(hits.match.value_counts().rename("entries").to_frame().T)
print("Names that match more than one player ID (must be 0, otherwise decide by hand):",
      (hits[hits.match == "matched"].groupby("name_key").player_id.nunique() > 1).sum())
print("Not found in the dataset (they did not play in 2022-26, so there is nothing to flag):",
      hits[hits.match == "NOT IN DATA"].player_name.tolist())

# 10d) CONSISTENCY CHECKS: does the data agree with the lists?
# (i) a retired player must not appear in matches AFTER his retirement year;
# (ii) a player who "opted out of IPL 2026" must not have played IPL 2026 (his current_team should be 'Not in IPL 2026').
hits["last_year"] = pd.to_datetime(hits.last_match).dt.year
bad_ret = hits[(hits.category.str.startswith("retired")) & (hits.last_year > hits.year_retired)]
print(f"Retired players seen playing after their retirement year: {len(bad_ret)}")
bad_opt = hits[(hits.status == "DOUBTFUL") & (hits.current_team != "Not in IPL 2026")]
print(f"'Opted out of IPL 2026' players who in fact have a 2026 IPL team in our data: {len(bad_opt)}")
display(hits[hits.match == "matched"][["player_name", "player_id", "category", "status", "age", "current_team", "last_match",
                                        "balls_batted_ipl", "balls_bowled_ipl"]].sort_values(["status", "category"]).round(1))

# 10e) AGE SANITY CHECK. The age file is only good to about +/- 1 year, but two retired veterans look badly wrong:
# Piyush Chawla (born 1988) shows as 28 and Amit Mishra (born 1982) as 22. Age drives RR's "young" preference, so we
# warn when a player's listed age is implausibly LOW for a long-serving player (we can only test the veterans we have a
# retirement list for, so this is a spot check, not a full audit).
vet = hits[(hits.category.str.startswith("retired")) & (hits.match == "matched")]
display(vet[vet.age < 30][["player_name", "age", "year_retired", "balls_batted_ipl", "balls_bowled_ipl"]])

# 10f) Attach the flags to the player master (one row per player; if a player is on both lists the stricter flag wins).
order = {"EXCLUDED": 2, "DOUBTFUL": 1}
g = (hits[hits.match == "matched"].assign(rank=lambda d: d.status.map(order))
         .sort_values("rank", ascending=False).drop_duplicates("player_id")
         [["player_id", "status", "category", "detail"]]
         .rename(columns={"status": "availability_guard", "category": "guard_reason", "detail": "guard_detail"}))
pm = pm.drop(columns=[c for c in ["availability_guard", "guard_reason", "guard_detail"] if c in pm.columns])
pm = pm.reset_index().merge(g, on="player_id", how="left").set_index("player_id")
pm["availability_guard"] = pm.availability_guard.fillna("OK")
pm = pm.drop(columns="name_key")
display(pm.availability_guard.value_counts().rename("players").to_frame().T)

# 10g) Save the guardrail table and the updated player master.
def save_table(df, name):
    try:
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False); return "parquet"
    except Exception:
        df.to_pickle(OUT_DIR / f"{name}.pkl"); return "pickle"
guard_out = hits[["player_name", "player_id", "category", "status", "last_ipl_team_stated", "year_retired", "detail", "source_file", "match"]]
print("Saved availability_guard:", guard_out.shape, save_table(guard_out, "availability_guard"))
player_master = pm.reset_index()
print("Saved player_master (with guardrail columns):", player_master.shape, save_table(player_master, "player_master"))

Guardrail entries: 20  ->  {'EXCLUDED': 16, 'DOUBTFUL': 4}


,matched,NOT IN DATA
entries,19,1


Names that match more than one player ID (must be 0, otherwise decide by hand): 0
Not found in the dataset (they did not play in 2022-26, so there is nothing to flag): ['Suresh Raina']
Retired players seen playing after their retirement year: 0
'Opted out of IPL 2026' players who in fact have a 2026 IPL team in our data: 0


,player_name,player_id,category,status,age,current_team,last_match,balls_batted_ipl,balls_bowled_ipl
16,Glenn Maxwell,838.000,"opted out of IPL 2026 auction, 2027 unconfirmed",DOUBTFUL,38.000,Not in IPL 2026,2025-11-06,488.000,380.000
17,Faf du Plessis,88.000,"opted out of IPL 2026 auction, 2027 unconfirmed",DOUBTFUL,42.000,Not in IPL 2026,2025-05-24,"1,276.000",0.000
18,Moeen Ali,"1,596.000","opted out of IPL 2026 auction, 2027 unconfirmed",DOUBTFUL,39.000,Not in IPL 2026,2025-05-07,380.000,443.000
19,Ben Stokes,"1,195.000","opted out of IPL 2026 auction, 2027 unconfirmed",DOUBTFUL,35.000,Not in IPL 2026,2023-04-03,14.000,6.000
14,Mohammad Amir,470.000,confirmed opt-out,EXCLUDED,34.000,Not in IPL 2026,2024-05-30,0.000,0.000
15,Ben Duckett,"2,379.000",ineligible after prior withdrawal (reported),EXCLUDED,32.000,Delhi Capitals,2026-02-03,0.000,0.000
0,Robin Uthappa,169.000,retired from IPL / all cricket,EXCLUDED,41.000,Not in IPL 2026,2022-05-12,171.000,0.000
2,Kieron Pollard,494.000,retired from IPL / all cricket,EXCLUDED,39.000,Not in IPL 2026,2022-05-09,134.000,83.000
3,Dwayne Bravo,161.000,retired from IPL / all cricket,EXCLUDED,43.000,Not in IPL 2026,2022-05-12,24.000,206.000
4,Ambati Rayudu,535.000,retired from IPL / all cricket,EXCLUDED,41.000,Not in IPL 2026,2023-05-29,337.000,0.000


,player_name,age,year_retired,balls_batted_ipl,balls_bowled_ipl
9,Piyush Chawla,28.000,"2,025.000",37.000,576.000
11,Amit Mishra,22.000,"2,025.000",30.000,126.000


,OK,EXCLUDED,DOUBTFUL
players,940,15,4


Saved availability_guard: (20, 9) parquet
Saved player_master (with guardrail columns): (959, 33) parquet


### Cell 10: Availability guardrails
**Procedure:** Read the two supplied files: `retired_players_IPL.csv` (14 players) and `auction_opt_outs.xlsx` (a sectioned sheet, parsed by section). Matched the names to `player_id` after lower-casing and stripping punctuation. Checked the lists against the data, then wrote a flag for each matched player into `player_master`. The strict flag is **EXCLUDED** (retired, confirmed opt-out, or ineligible). The soft flag is **DOUBTFUL** (skipped the 2026 auction, 2027 intent unconfirmed). Decision: flagged players stay in the model data, so the league baseline is not distorted. They are removed only from the final top-10 and auction shortlist.

**Observations**
- 20 entries: 16 excluded and 4 doubtful. 19 match a player in the data. **Suresh Raina** is not found because he last played before 2022, so there is nothing to flag.
- No name matches two player IDs, so the by-name match is unambiguous.
- **The lists agree with the data.** No retired player appears in a match after his retirement year, and none of the 4 "opted out of IPL 2026" players has a 2026 IPL team in our data.
- **Excluded:** the 14 retired players in the data (including Ashwin, Russell, Dhawan, Karthik, Saha, Pollard and Bravo), Mohammad Amir (confirmed opt-out) and Ben Duckett. Duckett's ineligibility comes from the supplied sheet (a reported two-season ban after withdrawing from the 2026 season) and is unverified. He still shows as a Delhi Capitals squad player in our current-team table.
- **Doubtful:** Maxwell (38), du Plessis (42), Moeen Ali (39) and Stokes (35), who last played IPL in 2025, 2025, 2025 and 2023. They stay available to the model but will carry an availability penalty in the auction layer.
- Final flags: 940 OK, 15 EXCLUDED, 4 DOUBTFUL.
- **Data caution (age file):** Piyush Chawla (listed 28) and Amit Mishra (listed 22) are clearly too young for veterans who retired in 2025, so the age file has errors. Ages drive RR's "young" preference, so every shortlisted player's age will be spot-checked against a public source before the final lists.
- Saved: `availability_guard` (20 rows) and the updated `player_master` (959 rows, 33 columns).